# Phase 7 — How Long Do the Pairs Live?

**The idea in one line.** Phase 6 re-picked the pairs every quarter. Here we ask: did it *need* to?
If the same pairs get picked quarter after quarter, relationships are durable and the re-selection is
cheap insurance; if the list churns constantly, relationships are short-lived and the re-selection is
doing the heavy lifting. Either answer is a finding.

**Why this matters for pairs trading.** The classic criticism of pairs trading is that relationships
break down exactly when you need them — in stressed markets. Instead of building a whole
market-regime model to test that, we do something simpler and quantitative: measure how much the pair
list *changes* at each rebalance (its **turnover**) and check whether change spikes when the market
gets rough. That gives a direct, defensible answer to the viva question *"do pair relationships
change during market stress?"*

**One accounting note.** A pair is the *unordered* couple of stocks. The Phase 4 test sometimes flips
which stock it calls "first" from quarter to quarter (it tries both and keeps the stronger direction);
that flip is the same economic pair, so we sort the two names before comparing quarters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

wf_pairs = pd.read_csv('data/model_input/wf_pairs.csv')
wf_summary = pd.read_csv('data/model_input/wf_summary.csv')
windows = pd.read_csv('data/model_input/window_dates.csv', index_col='window')
prices = pd.read_parquet('data/processed/adj_close_clean.parquet')

wf_pairs['pair'] = [ '|'.join(sorted([a, b])) for a, b in zip(wf_pairs.asset1, wf_pairs.asset2) ]
by_window = {w: set(g.pair) for w, g in wf_pairs.groupby('window')}
W = sorted(wf_summary.window)
print(f'{wf_pairs.pair.nunique()} distinct pairs traded across {len(W)} quarters')

## Survival, turnover, and lifetimes

Three simple measures, computed straight from the quarterly pair lists:

- **Survival rate** at each rebalance — of last quarter's pairs, what share was picked again this
  quarter?
- **Turnover** — the flip side: the share of last quarter's pairs that was *dropped* (1 − survival).
- **Lifetime** of a pair — how many consecutive quarters it stayed in the traded list before being
  dropped (a pair can have several separate "lives" if it returns later; we count each run).

In [ ]:
surv = {}
for prev, cur in zip(W[:-1], W[1:]):
    if by_window.get(prev):
        surv[cur] = len(by_window.get(cur, set()) & by_window[prev]) / len(by_window[prev])
persist = pd.DataFrame({'survival': pd.Series(surv)})
persist['turnover'] = 1 - persist.survival

runs = []
for pair in wf_pairs.pair.unique():
    present = [w for w in W if pair in by_window.get(w, set())]
    run = 1
    for a, b in zip(present[:-1], present[1:]):
        if b == a + 1:
            run += 1
        else:
            runs.append({'pair': pair, 'length': run}); run = 1
    runs.append({'pair': pair, 'length': run})
lifetimes = pd.DataFrame(runs)

print(f'Mean survival quarter-to-quarter: {persist.survival.mean():.0%}')
print(f'Mean lifetime: {lifetimes.length.mean():.1f} quarters | '
      f'longest-lived: {lifetimes.length.max()} quarters')
print(f'One-quarter wonders: {(lifetimes.length == 1).mean():.0%} of all pair-lives')
print(f'Long-lived (4+ consecutive quarters): {(lifetimes.length >= 4).sum()} pair-lives')

## The pair-lifetime picture

**Top — the pair timeline (a Gantt-style chart).** One row per distinct pair, one column per quarter;
a filled cell means "traded that quarter". Long horizontal streaks = durable relationships (expect
these among the big banks and utilities); scattered single cells = one-quarter wonders.

**Bottom left — lifetime distribution.** How many pair-lives lasted 1 quarter, 2 quarters, and so on.

**Bottom right — survival per rebalance.** The quarter-to-quarter survival rate over time; dips are
rebalances where the pair list was shaken up.

In [ ]:
pairs_sorted = (wf_pairs.groupby('pair').window.agg(['min', 'count'])
                .sort_values(['min', 'count']).index.tolist())
presence = pd.DataFrame(0, index=pairs_sorted, columns=W)
for w in W:
    for pair in by_window.get(w, set()):
        presence.loc[pair, w] = 1

fig = plt.figure(figsize=(13, 9))
ax0 = fig.add_subplot(2, 1, 1)
ax0.imshow(presence.values, aspect='auto', cmap='Blues', interpolation='nearest')
ax0.set_yticks(range(len(pairs_sorted))); ax0.set_yticklabels(pairs_sorted, fontsize=5)
ax0.set_xticks(range(len(W))); ax0.set_xticklabels(W, fontsize=7)
ax0.set_title('Pair lifetimes (dark = traded that quarter)'); ax0.set_xlabel('window')
ax1 = fig.add_subplot(2, 2, 3)
ax1.hist(lifetimes.length, bins=range(1, lifetimes.length.max() + 2), color='steelblue',
         rwidth=0.9, align='left')
ax1.set_title('Lifetime distribution'); ax1.set_xlabel('consecutive quarters'); ax1.set_ylabel('pair-lives')
ax2 = fig.add_subplot(2, 2, 4)
ax2.plot(persist.index, persist.survival, 'o-', color='seagreen')
ax2.set_ylim(0, 1.05); ax2.set_title('Quarter-to-quarter survival'); ax2.set_xlabel('window')
plt.tight_layout(); plt.show()

## Does the pair list churn more when markets are rough?

We need a "how rough was the market?" number for each rebalance. Rather than importing an external
fear gauge, we use the data we already have: the **realised volatility of the equal-weight market
over the quarter just ended** — literally how bumpy the average stock's ride was in the three months
before the rebalance (annualised so the number is recognisable). This is one of the proxies named in
the project plan, and using our own universe keeps every measurement inside the same 462 stocks.

Then one number answers the question: the **rank correlation** between that roughness and the
turnover of the pair list at the rebalance that followed. Positive and significant = stress really
does shake up pair relationships (the standard finding, and the reason quarterly re-selection earns
its keep); near zero = the pair list churns for idiosyncratic reasons instead.

**How to read the scatter:** each dot is one rebalance — market roughness just before it (left-right)
against how much of the pair list changed (up-down).

In [ ]:
mkt = prices.pct_change().mean(axis=1)
vol = {w: mkt.loc[windows.start_date[w - 1]:windows.end_date[w - 1]].std() * np.sqrt(252)
       for w in persist.index}
persist['prior_quarter_vol'] = pd.Series(vol)

rho, pval = spearmanr(persist.prior_quarter_vol, persist.turnover)
print(f'Spearman rank correlation (prior-quarter vol vs pair turnover): {rho:+.3f}  (p = {pval:.3f})')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(persist.prior_quarter_vol, persist.turnover, s=40, color='navy', alpha=0.7)
b, a = np.polyfit(persist.prior_quarter_vol, persist.turnover, 1)
xs = np.linspace(persist.prior_quarter_vol.min(), persist.prior_quarter_vol.max(), 20)
ax.plot(xs, a + b * xs, color='r')
ax.set_xlabel('realised vol of the market, prior quarter (annualised)')
ax.set_ylabel('pair-list turnover at the rebalance')
ax.set_title(f'Turnover vs market stress  (Spearman {rho:+.2f}, p={pval:.3f})')
plt.tight_layout(); plt.show()

## Save the hand-off files

- **persistence_summary.csv** — one row per rebalance: survival, turnover, prior-quarter volatility
  (Phase 10 draws the dissertation figure from this).
- **pair_lifetimes.csv** — every pair-life and its length (for the lifetime table in the write-up).

In [ ]:
persist.to_csv('data/model_input/persistence_summary.csv')
lifetimes.to_csv('data/model_input/pair_lifetimes.csv', index=False)
print('Saved persistence_summary.csv and pair_lifetimes.csv')

## Summary

We measured how durable the walk-forward's pair choices are: survival and turnover at each of the 26
rebalance transitions, the distribution of pair lifetimes, and — the punchline — whether turnover
tracks market stress, using the prior quarter's realised volatility as the stress gauge. The three
headline numbers for the write-up: **mean survival**, **share of one-quarter wonders**, and the
**turnover-vs-volatility correlation with its p-value**.